# 图
作为它的核心思想，LangGraph将工作流建模为图，可以使用三个核心部件定义agent的行为：

- State：表示应用程序当前快照的共享数据结构。它可以是任何数据类型，但通常使用共享状态模式定义。
- Node：对代理的逻辑进行编码的函数。它们接收当前状态作为输入，执行一些计算或副作用，并返回更新后的状态。
- Edge：根据当前状态确定下一步执行哪个节点的函数。它们可以是条件分支或固定转换。

节点负责工作，边负责判断下一个工作是什么。

LangGraph的底层图形算法使用消息传递来定义一个通用程序。当一个节点完成其操作时，它沿着一条或多条边向其他节点发送消息。然后，这些接收方节点执行它们的功能，将结果消息传递给下一组节点，然后继续这个过程。受谷歌的Pregel系统的启发，该程序以离散的“超级步骤”进行。


一个超级步骤可以看作是对图节点的一次迭代。并行运行的节点是同一个超级步骤的一部分，而顺序运行的节点属于单独的超级步骤。在图执行开始时，所有节点都处于非活动状态。当节点在其任何传入边（或“通道”）上接收到新消息（状态）时，节点变为活动状态。然后，活动节点运行其功能并以更新进行响应。在每个超级步骤结束时，没有传入消息的节点通过将自己标记为非活动来投票停止。当所有节点都处于非活动状态并且没有消息正在传输时，图执行终止。

要构建图，首先要定义状态，然后添加节点和边，最后对其进行编译。

编译是一个非常简单的步骤。它提供了对图结构的一些基本检查（没有孤立节点等）。您还可以在这里指定运行时参数，如检查指针和断点。你编译你的图只需调用compile方法：
```
graph = graph_builder.compile(...)
```
# State
定义图形时要做的第一件事是定义图形的状态。state由图的模式以及指定如何将更新应用于状态的reducer函数组成。State的模式将是图中所有node和edge的输入模式，可以是TypedDict或Pydantic模型。所有节点都会向状态发出更新，然后使用指定的reducer函数应用这些更新。
## 模式
指定图的模式的主要文档方法是使用TypedDict。如果您希望在您的状态中提供默认值，请使用数据类。如果你想要递归的数据验证，我们也支持使用Pydantic的BaseModel作为你的图状态（尽管Pydantic的性能不如TypedDict或数据类）。

默认情况下，图将具有相同的输入和输出模式。如果想要更改这一点，还可以直接指定显式输入和输出模式。当您有很多键，并且有些键显式用于输入，有些键显式用于输出时，这很有用。请参阅指南了解更多信息。

## 多种模式
通常，所有图节点都与一个模式通信。这意味着它们将对相同的状态通道进行读写。但是，在某些情况下，我们希望对此有更多的控制：

内部节点可以传递图的输入/输出中不需要的信息。

我们可能还想为图使用不同的输入/输出模式。例如，输出可能只包含一个相关的输出键。

可以让节点写入图中的私有状态通道以进行内部节点通信。我们可以简单地定义一个私有模式PrivateState。

还可以为图定义显式输入和输出模式。在这些情况下，我们定义了一个“内部”模式，其中包含与图操作相关的所有键。但是，我们还定义了输入和输出模式，它们是“内部”模式的子集，以约束图的输入和输出。

In [ ]:
from typing import Dict, Any, Optional, TypedDict
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
class InputState(TypedDict):
    user_input: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str

class PrivateState(TypedDict):
    bar: str

def node_1(state: InputState) -> OverallState:
    # Write to OverallState
    return {"foo": state["user_input"] + " name"}

def node_2(state: OverallState) -> PrivateState:
    # Read from OverallState, write to PrivateState
    return {"bar": state["foo"] + " is"}

def node_3(state: PrivateState) -> OutputState:
    # Read from PrivateState, write to OutputState
    return {"graph_output": state["bar"] + " Lance"}

builder = StateGraph(OverallState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
graph.invoke({"user_input":"My"})
# {'graph_output': 'My name is Lance'}

{'graph_output': 'My name is Lance'}

: 

这里有两个微妙而重要的要点需要注意：

1.我们将state: InputState作为输入模式传递给node_1。但是，我们写入foo，OverallState中的一个通道。我们如何向未包含在输入模式中的状态通道写入数据？这是因为节点可以写入图状态中的任何状态通道。图状态是初始化时定义的状态通道的联合，其中包括OverallState和过滤器InputState和OutputState。

2.我们使用如下的代码初始化图：
```
StateGraph(
    OverallState,
    input_schema=InputState,
    output_schema=OutputState
)
```
那么，我们如何在node_2中写入PrivateState呢？如果在StateGraph初始化中没有传递该模式，那么图如何获得对该模式的访问？

我们可以这样做，因为_nodes也可以声明额外的状态通道_，只要状态模式定义存在。在本例中，定义了PrivateState模式，因此我们可以将bar添加为图中的新状态通道并对其进行写入。
## Reducers
reducer是理解来自节点的更新如何应用于状态的关键。State中的每个键都有自己独立的reducer函数。如果没有显式指定reducer函数，则假定对该键的所有更新都应该覆盖它。有几种不同类型的减速器，从默认类型的reducer开始：

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    foo: int
    bar: list[str]

在本例中，没有为任何键指定减速器函数。假设图的输入为：

{"foo": 1, "bar": ["hi"]}。然后让我们假设第一个节点返回{"foo": 2}。这将被视为对状态的更新。注意，Node不需要返回整个State模式—只需一个更新即可。应用此更新后，状态将是{"foo": 2, "bar": ["hi"]}。如果第二个节点返回{"bar": ["bye"]}，那么状态将是{"foo": 2, "bar": ["bye"]}

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]

在本例中，我们使用Annotated类型为第二个键（bar）指定了一个reducer函数（operator.add）。注意，第一个键保持不变。假设图的输入是{"foo": 1, "bar": ["hi"]}。然后让我们假设第一个节点返回{"foo": 2}。这将被视为对状态的更新。注意，Node不需要返回整个State模式—只需一个更新即可。应用此更新后，状态将是{"foo": 2, "bar": ["hi"]}。如果第二个节点返回{"bar": ["bye"]}，那么state将是{"foo": 2, "bar": ["hi", "bye"]}。注意，这里通过将两个列表添加在一起来更新bar键。

在某些情况下，您可能希望绕过reducer并直接覆盖状态值。为此，LangGraph提供了Overwrite类型。
# 在图状态下处理消息
在许多情况下，将之前的对话历史记录存储为图状态中的消息列表是很有帮助的。为此，我们可以向存储messageobject列表的图形状态添加一个键（通道），并用一个reducer函数对其进行注释（参见下面示例中的messages key）。reducer函数对于告诉图如何在每次状态更新时更新状态中的Message对象列表（例如，当节点发送更新时）至关重要。如果没有指定reducer，则每次状态更新都会用最近提供的值覆盖消息列表。如果您想简单地将消息追加到现有列表中，则可以使用operator.add作为reducer。

然而，你可能也想在你的图形状态中手动更新消息（例如human-in- loop）。如果你要用operater.add，您发送到图的手动状态更新将被附加到现有消息列表中，而不是更新现有消息。为了避免这种情况，您需要一个可以跟踪消息id并在更新时覆盖现有消息的reducer。要实现这一点，可以使用预构建的add_messages函数。对于全新的消息，它将简单地附加到现有列表中，但它也将正确处理现有消息的更新。
## 序列化
除了跟踪消息id之外，只要在消息通道上接收到状态更新，add_messages函数还将尝试将消息反序列化到LangChain message对象中。

In [ ]:
# this is supported
{"messages": [HumanMessage(content="message")]}

# and this is also supported
{"messages": [{"type": "human", "content": "message"}]}

由于使用add_messages时状态更新总是被反序列化到LangChain消息中，因此应该使用点符号来访问消息属性，如state[" Messages "][-1].content。

In [ ]:
from langchain.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict

class GraphState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

## MessagesState
由于在您的状态中拥有消息列表是非常常见的，因此存在一个称为MessagesState的预构建状态，这使得使用消息变得容易。MessagesState使用单个消息键定义，该键是AnyMessage对象的列表，并使用add_messages reducer。通常，除了消息之外，还有更多的状态需要跟踪，所以我们看到人们子类化这个状态并添加更多字段，比如：

In [ ]:
from langgraph.graph import MessagesState

class State(MessagesState):
    documents: list[str]

# 节点
在LangGraph中，节点是Python函数（可以是同步的也可以是异步的），接受以下参数：
1. state -图的状态
2. 一个runnablecconfig对象，包含配置信息（如thread_id）和跟踪信息（如标签）
3. runtime -一个运行时对象，包含运行时上下文和其他信息，如store和stream_writer

In [ ]:
from dataclasses import dataclass
from typing_extensions import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph
from langgraph.runtime import Runtime

class State(TypedDict):
    input: str
    results: str

@dataclass
class Context:
    user_id: str

builder = StateGraph(State)

def plain_node(state: State):
    return state

def node_with_runtime(state: State, runtime: Runtime[Context]):
    print("In node: ", runtime.context.user_id)
    return {"results": f"Hello, {state['input']}!"}

def node_with_config(state: State, config: RunnableConfig):
    print("In node with thread_id: ", config["configurable"]["thread_id"])
    return {"results": f"Hello, {state['input']}!"}


builder.add_node("plain_node", plain_node)
builder.add_node("node_with_runtime", node_with_runtime)
builder.add_node("node_with_config", node_with_config)
...

在后台，函数被转换为RunnableLambda，这为您的函数添加了批处理和异步支持，以及本机跟踪和调试。

如果在不指定名称的情况下向图中添加节点，则将为其提供一个与函数名称等效的默认名称。

In [ ]:
builder.add_node(my_node)
# You can then create edges to/from this node by referencing it as `"my_node"`

START节点是一个特殊节点，它表示向图发送用户输入的节点。引用该节点的主要目的是确定应该首先调用哪个节点。

In [ ]:
from langgraph.graph import START

graph.add_edge(START, "node_a")

END节点是表示终端节点的特殊节点。当您想要表示哪些边在完成后没有任何动作时，将引用此节点。

In [ ]:
from langgraph.graph import END

graph.add_edge("node_a", END)

## 节点缓存
LangGraph支持基于节点输入的任务/节点缓存。使用缓存：

在编译图时指定缓存（或指定入口点）

为节点指定缓存策略。每个缓存策略支持：

Key_func用于根据节点的输入生成缓存键，默认为带有pickle的输入的散列。

Ttl，缓存的生存时间，以秒为单位。如果没有指定，缓存将永远不会过期。

In [ ]:
import time
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy


class State(TypedDict):
    x: int
    result: int


builder = StateGraph(State)


def expensive_node(state: State) -> dict[str, int]:
    # expensive computation
    time.sleep(2)
    return {"result": state["x"] * 2}


builder.add_node("expensive_node", expensive_node, cache_policy=CachePolicy(ttl=3))
builder.set_entry_point("expensive_node")
builder.set_finish_point("expensive_node")

graph = builder.compile(cache=InMemoryCache())

print(graph.invoke({"x": 5}, stream_mode='updates'))    
# [{'expensive_node': {'result': 10}}]
print(graph.invoke({"x": 5}, stream_mode='updates'))    
# [{'expensive_node': {'result': 10}, '__metadata__': {'cached': True}}]

第一次运行需要两秒钟（由于模拟的昂贵计算）。

第二次运行利用缓存并快速返回。

# 边
边定义了逻辑如何路由以及图形如何决定停止。这是代理如何工作以及不同节点如何相互通信的重要部分。

一个节点可以有多个出边。如果一个节点有多个出边，所有这些目标节点将作为下一个超级步骤的一部分并行执行。
## 自然边
如果总是想从节点A到节点B，可以直接使用add_edge方法。

In [ ]:
graph.add_edge("node_a", "node_b")

## 条件边
如果您希望有选择地路由到一个或多个边缘（或有选择地终止），您可以使用add_conditional_edges方法。这个方法接受一个节点的名称和一个“路由函数”，在该节点被执行后调用：

In [ ]:
graph.add_conditional_edges("node_a", routing_function)

与节点类似，routing_function接受图的当前状态并返回一个值。

默认情况下，返回值routing_function被用作发送状态到next的节点（或节点列表）的名称。作为下一个超级步骤的一部分，所有这些节点将并行运行。

您可以选择提供一个字典，将routing_function的输出映射到下一个节点的名称。

如果你想在一个函数中合并状态更新和路由，请使用Command而不是条件边。
## 入口点
入口点是图启动时运行的第一个节点。可以从虚拟START节点到要执行的第一个节点使用add_edgemmethod来指定在哪里输入图。

In [ ]:
from langgraph.graph import START

graph.add_edge(START, "node_a")

## 条件入口点
条件入口点允许您根据自定义逻辑从不同的节点开始。您可以使用虚拟START节点中的add_conditional_edges来完成此操作。

In [ ]:
from langgraph.graph import START

graph.add_conditional_edges(START, routing_function)

您可以选择提供一个字典，将routing_function的输出映射到下一个节点的名称。

In [ ]:
graph.add_conditional_edges(START, routing_function, {True: "node_b", False: "node_c"})

默认情况下，节点（Nodes）和边（Edges）都是预先定义好的，并且操作在同一个共享状态（State）上。然而，在某些情况下，你可能无法预先知道确切的边，或者可能希望同时存在不同版本的状态。这种场景的一个常见例子是分治（map-reduce）设计模式。在该模式中，一个节点可能会生成一个对象列表，而你希望将另一个节点应用到列表中的每个对象上。对象的数量可能是预先未知的（这意味着边的数量也可能未知），且下游节点的输入状态应该各不相同（每个生成的对象对应一个独立状态）。
为了支持这种设计模式，LangGraph 允许从条件边中返回 Send 对象。Send 接收两个参数：第一个是目标节点的名称，第二个是要传递给该节点的状态。

In [ ]:
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state['subjects']]

graph.add_conditional_edges("node_a", continue_to_jokes)

将控制流（边）和状态更新（节点）结合起来是很有用的。例如，您可能既要执行状态更新，又要决定在同一节点中下一步转到哪个节点。LangGraph提供了一种从节点函数返回Command对象的方法：

In [ ]:
def my_node(state: State) -> Command[Literal["my_other_node"]]:
    return Command(
        # state update
        update={"foo": "bar"},
        # control flow
        goto="my_other_node"
    )

使用命令，您还可以实现动态控制流行为（与条件边相同）：

In [ ]:
def my_node(state: State) -> Command[Literal["my_other_node"]]:
    if state["foo"] == "bar":
        return Command(update={"foo": "baz"}, goto="my_other_node")

当在节点函数中返回Command时，你必须添加返回类型注释，其中包含节点路由到的节点名称列表，例如Command[Literal["my_other_node"]]。这对于图形呈现是必需的，并告诉LangGraph my_node可以导航到my_other_node。

当您需要更新图状态和到不同节点的路由时，使用Command。例如，在实现多代理切换时，路由到不同的代理并将一些信息传递给该代理是很重要的。

使用条件边在节点之间有条件地路由，而不更新状态。
## 导航到父图中的一个节点
如果你正在使用子图，你可能想从子图中的一个节点导航到另一个子图（即父图中的另一个节点）。为此，可以指定graph=Command。

In [ ]:
def my_node(state: State) -> Command[Literal["other_subgraph"]]:
    return Command(
        update={"foo": "bar"},
        goto="other_subgraph",  # where `other_subgraph` is a node in the parent graph
        graph=Command.PARENT
    )

设置graph为Command.PARENT将导航到最近的父图。
当您为父图状态模式和子图状态模式共享的键从子图节点发送更新到父图节点时，您必须为在父图状态中更新的键定义一个reducer。
## 运行时上下文
在创建图时，可以为传递给节点的运行时上下文指定一个context_schema。这对于将信息传递给不属于图状态的节点非常有用。例如，您可能希望传递诸如模型名称或数据库连接之类的依赖项。

In [ ]:
@dataclass
class ContextSchema:
    llm_provider: str = "openai"

graph = StateGraph(State, context_schema=ContextSchema)

然后，您可以使用invoemethod的context参数将这个上下文传递到图中。

In [ ]:
graph.invoke(inputs, context={"llm_provider": "anthropic"})

然后你可以在节点或条件边中访问和使用这个上下文：

In [ ]:
from langgraph.runtime import Runtime

def node_a(state: State, runtime: Runtime[ContextSchema]):
    llm = get_llm(runtime.context.llm_provider)
    # ...

## 递归限制
递归限制设置图在单次执行期间可以执行的最大超级步骤数。一旦达到限制，LangGraph将引发GraphRecursionError。默认情况下，该值设置为25步。可以在运行时在任何图上设置递归限制，并通过配置字典传递给invoke/stream。重要的是，recursion_limit是一个独立的配置键，不应该像所有其他用户定义的配置一样在可配置键中传递。请看下面的例子：

In [ ]:
graph.invoke(inputs, config={"recursion_limit": 5}, context={"llm": "anthropic"})